# الدرس 01 - مقدمة لوكلاء الذكاء الاصطناعي

مرحبًا بك في الدرس الأول من دورة **وكلاء الذكاء الاصطناعي للمبتدئين**!

الـ **وكيل الذكاء الاصطناعي** هو برنامج يستخدم نموذج لغة كبير (LLM) كمحرك استدلال ويمكنه اتخاذ *إجراءات* في العالم الحقيقي — مثل استدعاء واجهات برمجة التطبيقات، والاستعلام من قواعد البيانات، أو تشغيل الشيفرة — لتحقيق هدف نيابةً عن المستخدم.

في هذه الدفترية ستقوم ببناء وكيلك الأول: **وكيل السفر** الذي يوصي بوجهات العطلات. على طول الطريق ستتعلم كيف:

1. الاتصال بخدمة Azure AI Foundry Agent باستخدام **إطار عمل الوكيل من مايكروسوفت**.
2. إعطاء الوكيل **أداة** — دالة بايثون بسيطة يمكنه استدعاؤها.
3. تشغيل الوكيل وفحص استجابته.
4. تدفق استجابة الوكيل رمزًا برمز.


## الإعداد

قبل تشغيل هذا الدفتر، تأكد من أنك تمتلك:

1. **مشروع Azure AI Foundry** مع نموذج محادثة نشرته (مثلاً `gpt-4o-mini`).
2. **تسجيل الدخول باستخدام Azure CLI** — شغّل الأمر `az login` في نافذة الطرفية.
3. **تعيين متغيرات البيئة المطلوبة:**
   - `AZURE_AI_PROJECT_ENDPOINT` — نقطة النهاية لمشروع Azure AI Foundry الخاص بك.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — اسم النموذج الذي نشرته.

الخلية أدناه تثبت حزم بايثون التي تحتاجها.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## إنشاء وكيلك الأول

يحتاج الوكيل إلى شيئين:

- **تعليمات** تخبره *من هو* و*كيف يتصرف* (موجه النظام).
- **أدوات** — وظائف Python مزينة بـ `@tool` يمكن للوكيل استدعاؤها لاسترداد المعلومات أو تنفيذ الإجراءات.

في ما يلي نُعرّف أداة بسيطة تُرجع قائمة بأشهر وجهات العطلات. سيستخدم الوكيل هذه الأداة عندما يطلب المستخدم توصيات سفر.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## الردود المتدفقة

لتجربة أكثر تفاعلية يمكنك **بث** رد الوكيل. بدلاً من الانتظار للرد الكامل، يقوم الوكيل بإصدار أجزاء النص أثناء توليدها. هذا مفيد بشكل خاص في واجهات الدردشة حيث تريد عرض المخرجات في الوقت الحقيقي.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## الملخص

في هذا الدرس تعلمت كيفية:

- **إنشاء مزود** يتصل بخدمة Azure AI Foundry Agent عبر `AzureAIProjectAgentProvider`.
- **تعريف أداة** باستخدام المزخرف `@tool` حتى يتمكن الوكيل من استدعاء دوال بايثون الخاصة بك.
- **تشغيل الوكيل** مع رسالة المستخدم وطباعة رده.
- **بث الردود** للإخراج في الوقت الفعلي.

في الدرس التالي سنستكشف أطر العمل الوكيلية بمزيد من العمق ونتعلم كيفية منح الوكلاء أدوات أكثر قوة وقدرات استدلال متعددة الخطوات.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**إخلاء المسؤولية**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى لتحقيق الدقة، يرجى العلم بأن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الموثوق به. للمعلومات الحساسة والهامة، يُنصح بالترجمة الاحترافية من قبل مترجم بشري. نحن غير مسؤولين عن أي سوء فهم أو تفسيرات خاطئة ناتجة عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
